In [1]:
from dotenv import load_dotenv
# from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_groq import ChatGroq
from langchain.tools import tool
from langchain.agents import create_agent

load_dotenv()

True

In [2]:
PRODUCTS = {
    "wireless headphones": {"price": 79.99,  "rating": 4.6, "description": "Over-ear Bluetooth, 30-hr battery, active noise cancellation."},
    "smart watch":         {"price": 199.99, "rating": 4.3, "description": "Tracks heart rate and sleep. 5-day battery, water-resistant."},
    "mechanical keyboard": {"price": 129.00, "rating": 4.8, "description": "Tenkeyless, Cherry MX Brown switches, per-key RGB."},
    "laptop stand":        {"price": 34.99,  "rating": 4.5, "description": "Adjustable aluminium, fits 11-17 inch laptops, folds flat."},
}

@tool
def get_product(name: str) -> str:
    """Look up a product by name and return its price, rating, stock, and description."""
    p = PRODUCTS.get(name.lower())
    if not p:
        return f"Product not found. Available: {', '.join(PRODUCTS)}"
    return str(p)

In [3]:
# llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0)
llm = ChatGroq(model="openai/gpt-oss-20b", temperature=0)

agent = create_agent(
    llm,
    tools=[get_product],
    system_prompt="You are a helpful product assistant for an online tech store.",
)

In [4]:
def ask(question: str):
    result = agent.invoke({"messages": [{"role": "user", "content": question}]})
    print(result["messages"][-1].content)

In [5]:
ask("What is the price of the wireless headphones.")

The wireless headphones are priced at **$79.99**.


In [6]:
REVIEWS = {
    "wireless headphones": {"reviews": 1262, "rating": 4.6},
    "smart watch":         {"reviews": 340,  "rating": 3.9},
    "mechanical keyboard": {"reviews": 67,   "rating": 4.8},
    "laptop stand":        {"reviews": 781,  "rating": 4.5},
}

@tool
def get_review(name: str) -> str:
    """Look up a product review by a product name. Return the product name, number of reviews and rating"""
    r = REVIEWS.get(name.lower())
    if not r:
        return f"Review not available for this product"
    return str(r)

In [7]:
# llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0)
llm = ChatGroq(model="openai/gpt-oss-20b", temperature=0)

agent2 = create_agent(
    llm,
    tools=[get_product, get_review],
    system_prompt="You are a helpful product assistant for an online tech store.",
)

def ask2(question: str):
    result = agent2.invoke({"messages": [{"role": "user", "content": question}]})
    print(result["messages"][-1].content)

In [8]:
ask2("how do people like smart watch")

Sure! Could you let me know which smart‑watch brand or model you’re interested in? That way I can pull up the latest reviews and give you a clear picture of what people are saying.


In [9]:
ask2("what is the price and reviews of smart watch")

**Smart Watch**

- **Price:** $199.99  
- **Reviews:** 340 reviews (average rating 3.9 / 5)


In [10]:
ask2("What are the reviews on this products") # Thus we can see that LLM doesn't persist the previous interactions

I’m happy to help! Could you let me know which product you’re interested in?


In [11]:
from langgraph.checkpoint.memory import InMemorySaver


In [12]:
llm = ChatGroq(model="openai/gpt-oss-20b", temperature=0)

agent2 = create_agent(
    llm,
    tools=[get_product, get_review],
    system_prompt="You are a helpful product assistant for an online tech store.",
    checkpointer=InMemorySaver()
)

def ask3(question: str):
    config = {"configurable": {"thread_id": "user-alice-session-1"}}
    result = agent2.invoke(
        {
            "messages": [
                {"role": "user", "content": question}
            ] 
        },
        config=config
    )
    print(result["messages"][-1].content)

In [13]:
ask3("what is the price of wireless headphones.")

The wireless headphones are priced at **$79.99**.


In [14]:
ask3("What are the review on this product")

Here’s what customers are saying about the wireless headphones:

- **Number of reviews:** 1,262  
- **Average rating:** 4.6 / 5

Feel free to let me know if you’d like more details, such as top positive or negative comments!
